Cover ablation: how much does the main model change with and without the cover block V?

Question: on the same set of serials, does adding the cover vector V to the main model's features S+A+T change how well it predicts top10?

Method follows the repo's `modeling/` exactly:
- Features come straight from the repo's `features.py` (S, A, T), not rebuilt here;
- Logistic regression with standardization; C picked by AUC on the 2022 validation year; refit on 2017–2022; 2023 test year scored once, 2024 as a supplement;
- Metrics: AUC, AP, Recall@slots; differences get 95% intervals from a paired bootstrap (resampling within each quarter, 500 draws).

The only difference: every comparison uses **serials with a valid cover** only (`status == "valid"`), and all models use the same serials.

How to run: open in Colab → a CPU runtime is enough (no GPU) → Run all. About 10–15 minutes. Results go to `06_ablation/` on Drive.

In [1]:
import os, sys, json, time, shutil, subprocess, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

REPO_URL = "https://github.com/godspeedhuang/royalroad-breakout"
HF_REPO = "syuanbo/royalroad-breakout"
CS = [1e-4, 1e-3, 1e-2, 1e-1]          # same as the repo's 01_ml.py
B = 500                                 # same as the repo's 03_compare.py
SEED = 0
REPO_MAIN_TEST_AUC = 0.887              # test AUC of lr_SAT in the repo README, used to check the reproduction

if os.environ.get("ABL_LOCAL_ROOT"):    # for local testing
    DRIVE = Path(os.environ["ABL_LOCAL_ROOT"])
    WORK = Path(os.environ["ABL_LOCAL_WORK"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
    WORK = Path("/content")

COVER = DRIVE / "RR_project1" / "cover_v2" / "封面向量_V区块_交接"   # Drive folder name (Chinese), kept as is
if not COVER.exists():                  # in case the folder was renamed back
    COVER = DRIVE / "RR_project1" / "cover_v2" / "full"
OUT = COVER / "06_ablation"
GH_OUT = COVER / "05_for_github" / "out" / "ablation"
for d in [OUT, GH_OUT]:
    d.mkdir(parents=True, exist_ok=True)
REPO = WORK / "royalroad-breakout"

T0 = time.time()
def log(msg):
    line = f"[{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S UTC}] {msg}"
    print(line, flush=True)
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    (OUT / "progress.json").write_text(json.dumps({"last": msg, "elapsed_min": round((time.time() - T0) / 60, 1)}, ensure_ascii=False))

def check(cond, step, expected, actual):
    if not cond:
        msg = f"[CHECK FAILED] step: {step} | expected: {expected} | actual: {actual}"
        log(msg)
        raise RuntimeError(msg)

log(f"Start. Cover folder: {COVER}")
check((COVER / "01_status" / "cover_status.parquet").exists(), "Step 1 environment", "cover_status.parquet found on Drive", f"not found under {COVER}")

Mounted at /content/drive
[2026-11-14 23:07:05 UTC] Start. Cover folder: /content/drive/MyDrive/RR_project1/cover_v2/封面向量_V区块_交接


Step 1: get the repo's code and feature file.

In [2]:
if not (REPO / "modeling" / "features.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
log(f"Repo commit: {COMMIT}")

_books = REPO / "embedding" / "out" / "books.parquet"
if not _books.exists():
    from huggingface_hub import hf_hub_download
    hf_hub_download(HF_REPO, "embedding/out/books.parquet", repo_type="dataset", local_dir=str(REPO))
check(_books.exists(), "Step 1", "books.parquet downloaded", "missing")
check((REPO / "embedding" / "out" / "watermark_books.csv").exists(), "Step 1", "watermark_books.csv is in the repo", "missing")

sys.path.insert(0, str(REPO / "modeling"))
import features as F
b, X = F.load()
sp = F.splits(b)
log(f"Repo features loaded: {len(b)} serials; " + ", ".join(f"{k} {v.shape[1]} dims" for k, v in X.items()))
check(len(b) == 45245 or os.environ.get("ABL_LOCAL_ROOT"), "Step 1", "45,245 serials", len(b))
check(all(X[k].shape[0] == len(b) for k in X), "Step 1", "every block has one row per serial", {k: v.shape for k, v in X.items()})

[2026-11-14 23:07:12 UTC] Repo commit: 2e1d68a93fccf29ed1268c91079083ddc3ad43c5


embedding/out/books.parquet: reconstructing file:   0%|          |  0.00B /  866MB            

embedding/out/books.parquet: downloading bytes:           |  0.00B            

[2026-11-14 23:08:04 UTC] Repo features loaded: 45245 serials; D 1537 dims, T 1536 dims, C 1536 dims, S 75 dims, A 2 dims, P 1 dims


Step 2: load the cover block V, aligned to the repo's serial order.

In [3]:
def load_cover_block(fiction_ids, status_path, emb_path):
    """Same as the loading example in the README. Returns (V, cover_valid)."""
    st = pd.read_parquet(status_path, columns=["fiction_id", "status"])
    t = pq.read_table(emb_path)
    ids = t.column("fiction_id").to_numpy()
    E = t.column("emb").combine_chunks().flatten().to_numpy().reshape(len(ids), -1).astype(np.float32)
    fiction_ids = np.asarray(fiction_ids)
    cover_valid = np.isin(fiction_ids, st.fiction_id[st.status == "valid"].values)
    pos = pd.Series(np.arange(len(ids)), index=ids).reindex(fiction_ids[cover_valid]).values.astype(int)
    V = np.zeros((len(fiction_ids), E.shape[1]), dtype=np.float32)
    V[cover_valid] = E[pos]
    return V, cover_valid

STATUS = COVER / "01_status" / "cover_status.parquet"
EMBS = {"clip_l14": 768, "clip_b32": 512, "dinov2_base": 768, "resnet50": 2048}
valid = None
for name, dim in EMBS.items():
    X[f"V_{name}"], cv = load_cover_block(b.fiction_id.values, STATUS, COVER / "03_embeddings" / f"cover_emb_{name}.parquet")
    check(X[f"V_{name}"].shape == (len(b), dim), "Step 2", f"{name} shape ({len(b)}, {dim})", X[f"V_{name}"].shape)
    valid = cv if valid is None else valid
    check((cv == valid).all(), "Step 2", "the 4 embeddings cover the same valid serials", "mismatch")

_st = pd.read_parquet(COVER / "03_embeddings" / "cover_stats.parquet")
_st = _st.set_index("fiction_id").reindex(b.fiction_id.values)
X["V_stats"] = np.nan_to_num(_st.values.astype(np.float32))
check(not np.isnan(X["V_stats"][valid]).any(), "Step 2", "every valid serial has visual statistics", "missing values")

# Check: the repo's year / top10 match the cover status table (guards against misaligned rows)
_cs = pd.read_parquet(STATUS, columns=["fiction_id", "year", "top10"]).set_index("fiction_id").reindex(b.fiction_id.values)
check((_cs.year.values == b.year.values).all() and (_cs.top10.values.astype(bool) == b.top10.values.astype(bool)).all(),
      "Step 2", "year and top10 in the cover status table match the repo row by row", "mismatch, rows may be misaligned")

b["cover_valid"] = valid
n_split = {k: int((m & valid).sum()) for k, m in sp.items()}
log(f"Valid-cover serials: {int(valid.sum())} in total; by split {n_split}")
if not os.environ.get("ABL_LOCAL_ROOT"):
    check(n_split == {"train": 4236, "val": 2927, "test": 4507, "recent": 7335}, "Step 2",
          "matches the cover README: train 4236 / val 2927 / test 4507 / recent 7335", n_split)

[2026-11-14 23:08:26 UTC] Valid-cover serials: 19006 in total; by split {'train': 4236, 'val': 2927, 'test': 4507, 'recent': 7335}


Step 3: train and score.

Each model: fit every C on the training years (2017–2021), pick C by AUC on the validation year (2022), refit on 2017–2022, then score 2023 and 2024.

In [4]:
y, q = b.top10.values.astype(int), b.quarter.values
lr = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))

def run(Z, mask, shuffle=False):
    tr, va = sp["train"] & mask, sp["val"] & mask
    trva, ev = tr | va, (sp["test"] | sp["recent"]) & mask
    ytr = np.random.default_rng(SEED).permutation(y[tr]) if shuffle else y[tr]
    val = {C: F.metrics(y[va], lr(C).fit(Z[tr], ytr).decision_function(Z[va]), q[va]) for C in CS}
    C = max(val, key=lambda c: val[c]["auc"])
    ytrva = np.random.default_rng(SEED).permutation(y[trva]) if shuffle else y[trva]
    score = np.full(len(b), np.nan)
    score[ev] = lr(C).fit(Z[trva], ytrva).decision_function(Z[ev])
    return C, val[C], score

BLOCKS = {                                   # name → which blocks to use
    "SAT": ["S", "A", "T"],
    "SAT+V_clip_l14": ["S", "A", "T", "V_clip_l14"],
    "SAT+V_clip_b32": ["S", "A", "T", "V_clip_b32"],
    "SAT+V_dinov2_base": ["S", "A", "T", "V_dinov2_base"],
    "SAT+V_resnet50": ["S", "A", "T", "V_resnet50"],
    "SAT+V_stats": ["S", "A", "T", "V_stats"],
    "SA": ["S", "A"],
    "SA+V_clip_l14": ["S", "A", "V_clip_l14"],
    "V_clip_l14": ["V_clip_l14"],
    "V_clip_b32": ["V_clip_b32"],
    "V_dinov2_base": ["V_dinov2_base"],
    "V_resnet50": ["V_resnet50"],
    "V_stats": ["V_stats"],
}

rows, scores = [], {}
test_v, recent_v = sp["test"] & valid, sp["recent"] & valid
for name, blks in BLOCKS.items():
    t = time.time()
    C, v, s = run(np.hstack([X[k] for k in blks]), valid)
    scores[name] = s
    te, re_ = F.metrics(y[test_v], s[test_v], q[test_v]), F.metrics(y[recent_v], s[recent_v], q[recent_v])
    rows.append({"model": name, "sample": "valid_cover", "C": C,
                 **{f"val_{k}": v_ for k, v_ in v.items()}, **{f"test_{k}": v_ for k, v_ in te.items()},
                 **{f"y2024_{k}": v_ for k, v_ in re_.items()}})
    log(f"{name:20s} C={C:<7g} val AUC {v['auc']:.3f}  test AUC {te['auc']:.3f}  test Recall@slots {te['recall_at_slots']:.3f} ({time.time() - t:.0f} s)")

# Control 1: shuffled training labels (should be near 0.5; confirms rows are not misaligned)
C, v, s = run(np.hstack([X[k] for k in BLOCKS["SAT+V_clip_l14"]]), valid, shuffle=True)
SHUFFLED_TEST_AUC = F.metrics(y[test_v], s[test_v], q[test_v])["auc"]
log(f"SAT+V_clip_l14 with shuffled labels: test AUC {SHUFFLED_TEST_AUC:.3f}")

# Control 2: SAT trained and tested on all serials should reproduce the repo main model's test AUC of 0.887
C, v, s = run(np.hstack([X[k] for k in BLOCKS["SAT"]]), np.ones(len(b), bool))
te_all = F.metrics(y[sp["test"]], s[sp["test"]], q[sp["test"]])
rows.append({"model": "SAT", "sample": "all_books", "C": C, **{f"val_{k}": v_ for k, v_ in v.items()},
             **{f"test_{k}": v_ for k, v_ in te_all.items()}})
log(f"SAT on all serials (reproducing the repo main model): test AUC {te_all['auc']:.3f} (repo reports {REPO_MAIN_TEST_AUC})")

res = pd.DataFrame(rows)
res.to_csv(OUT / "ablation_results.csv", index=False)

[2026-11-14 23:08:35 UTC] SAT                  C=0.001   val AUC 0.862  test AUC 0.858  test Recall@slots 0.289 (9 s)
[2026-11-14 23:08:42 UTC] SAT+V_clip_l14       C=0.001   val AUC 0.874  test AUC 0.870  test Recall@slots 0.360 (7 s)
[2026-11-14 23:08:50 UTC] SAT+V_clip_b32       C=0.001   val AUC 0.864  test AUC 0.868  test Recall@slots 0.344 (8 s)
[2026-11-14 23:08:58 UTC] SAT+V_dinov2_base    C=0.001   val AUC 0.863  test AUC 0.858  test Recall@slots 0.328 (8 s)
[2026-11-14 23:09:10 UTC] SAT+V_resnet50       C=0.001   val AUC 0.853  test AUC 0.851  test Recall@slots 0.304 (12 s)
[2026-11-14 23:09:17 UTC] SAT+V_stats          C=0.001   val AUC 0.862  test AUC 0.858  test Recall@slots 0.289 (6 s)
[2026-11-14 23:09:17 UTC] SA                   C=0.01    val AUC 0.806  test AUC 0.782  test Recall@slots 0.269 (1 s)
[2026-11-14 23:09:20 UTC] SA+V_clip_l14        C=0.001   val AUC 0.809  test AUC 0.808  test Recall@slots 0.324 (2 s)
[2026-11-14 23:09:22 UTC] V_clip_l14           C=0.0001

Step 4: paired bootstrap, how much covers add (2023 test year).

In [5]:
rng = np.random.default_rng(SEED)
tix = np.flatnonzero(test_v)
by_q = [tix[q[tix] == qq] for qq in np.unique(q[tix])]
PAIRS = [("SAT+V_clip_l14", "SAT"), ("SAT+V_clip_b32", "SAT"), ("SAT+V_dinov2_base", "SAT"),
         ("SAT+V_resnet50", "SAT"), ("SAT+V_stats", "SAT"), ("SA+V_clip_l14", "SA")]
draws = {p: {"auc": np.empty(B), "ap": np.empty(B), "recall_at_slots": np.empty(B)} for p in PAIRS}
for i in range(B):
    ix = np.concatenate([rng.choice(g, len(g)) for g in by_q])
    for a, c in PAIRS:
        ma, mc = F.metrics(y[ix], scores[a][ix], q[ix]), F.metrics(y[ix], scores[c][ix], q[ix])
        for k in ma:
            draws[(a, c)][k][i] = ma[k] - mc[k]
    if (i + 1) % 100 == 0:
        log(f"bootstrap {i + 1}/{B}")

point = res[res["sample"] == "valid_cover"].set_index("model")
brows = []
for (a, c), d in draws.items():
    for k, v in d.items():
        brows.append({"with_cover": a, "without_cover": c, "metric": k,
                      "delta": point.loc[a, f"test_{k}"] - point.loc[c, f"test_{k}"],
                      "ci_low": np.percentile(v, 2.5), "ci_high": np.percentile(v, 97.5)})
boot = pd.DataFrame(brows)
boot.to_csv(OUT / "ablation_bootstrap.csv", index=False)
print(boot.round(4).to_string(index=False))

[2026-11-14 23:10:25 UTC] bootstrap 100/500
[2026-11-14 23:10:43 UTC] bootstrap 200/500
[2026-11-14 23:11:01 UTC] bootstrap 300/500
[2026-11-14 23:11:19 UTC] bootstrap 400/500
[2026-11-14 23:11:37 UTC] bootstrap 500/500
       with_cover without_cover          metric   delta  ci_low  ci_high
   SAT+V_clip_l14           SAT             auc  0.0120  0.0036   0.0200
   SAT+V_clip_l14           SAT              ap  0.0343  0.0155   0.0549
   SAT+V_clip_l14           SAT recall_at_slots  0.0711  0.0273   0.1074
   SAT+V_clip_b32           SAT             auc  0.0099  0.0028   0.0165
   SAT+V_clip_b32           SAT              ap  0.0261  0.0088   0.0447
   SAT+V_clip_b32           SAT recall_at_slots  0.0553  0.0151   0.0889
SAT+V_dinov2_base           SAT             auc -0.0007 -0.0095   0.0066
SAT+V_dinov2_base           SAT              ap  0.0307  0.0131   0.0507
SAT+V_dinov2_base           SAT recall_at_slots  0.0395  0.0037   0.0815
   SAT+V_resnet50           SAT             auc -0

Step 5: figure, acceptance checks, export.

In [6]:
order = ["SAT", "SAT+V_clip_l14", "SAT+V_clip_b32", "SAT+V_dinov2_base", "SAT+V_resnet50", "SAT+V_stats",
         "SA", "SA+V_clip_l14", "V_clip_l14", "V_clip_b32", "V_dinov2_base", "V_resnet50", "V_stats"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
for ax, k, title in [(axes[0], "test_auc", "Test AUC (2023)"), (axes[1], "test_recall_at_slots", "Test Recall@slots (2023)")]:
    vals = point.loc[order, k]
    ax.barh(range(len(order)), vals, color=["#2a78d6" if m.startswith("SAT") else "#7f8c8d" if m.startswith("SA") else "#eb6834" for m in order])
    for i_, v_ in enumerate(vals):
        ax.text(v_, i_, f" {v_:.3f}", va="center", fontsize=9)
    ax.set_title(title); ax.set_xlim(0, max(1.0, vals.max() * 1.15))
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order); axes[0].invert_yaxis()
fig.suptitle(f"Cover ablation on books with a valid cover (test n = {int(test_v.sum())}, positives = {int(y[test_v].sum())})")
plt.tight_layout(); plt.savefig(OUT / "ablation.png", dpi=130); plt.close()

CHECKS = []
def add(name, ok, detail):
    CHECKS.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})
    print(("PASS  " if ok else "FAIL  ") + name + " | " + str(detail))
add("SAT on all serials reproduces the repo main model (test AUC within 0.01)", abs(te_all["auc"] - REPO_MAIN_TEST_AUC) < 0.01, f"{te_all['auc']:.3f} vs {REPO_MAIN_TEST_AUC}")
add("test AUC with shuffled labels is near 0.5 (0.45–0.55)", 0.45 <= SHUFFLED_TEST_AUC <= 0.55, f"{SHUFFLED_TEST_AUC:.3f}")
add("all models use the same serials", True, f"train {n_split['train']} / val {n_split['val']} / test {n_split['test']} / 2024 {n_split['recent']}")
add("C picked on the validation year only; test year scored once", True, "follows the repo's 01_ml.py")
summary = {"repo_commit": COMMIT, "n_valid_by_split": n_split, "test_positives": int(y[test_v].sum()),
           "test_top10_rate": float(y[test_v].mean()), "checks": CHECKS,
           "all_pass": all(c["result"] == "PASS" for c in CHECKS)}
(OUT / "ablation_checks.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")

for f in ["ablation_results.csv", "ablation_bootstrap.csv", "ablation_checks.json", "ablation.png", "run_log.txt"]:
    shutil.copy(OUT / f, GH_OUT / f)
log(f"All done: {sum(c['result'] == 'PASS' for c in CHECKS)}/{len(CHECKS)} checks passed. Results in {OUT}")
print(point.loc[order, ["C", "val_auc", "test_auc", "test_ap", "test_recall_at_slots"]].round(3).to_string())

PASS  SAT on all serials reproduces the repo main model (test AUC within 0.01) | 0.887 vs 0.887
PASS  test AUC with shuffled labels is near 0.5 (0.45–0.55) | 0.509
PASS  all models use the same serials | train 4236 / val 2927 / test 4507 / 2024 7335
PASS  C picked on the validation year only; test year scored once | follows the repo's 01_ml.py
[2026-11-14 23:11:37 UTC] All done: 4/4 checks passed. Results in /content/drive/MyDrive/RR_project1/cover_v2/封面向量_V区块_交接/06_ablation
                       C  val_auc  test_auc  test_ap  test_recall_at_slots
model                                                                     
SAT                0.001    0.862     0.858    0.269                 0.289
SAT+V_clip_l14     0.001    0.874     0.870    0.303                 0.360
SAT+V_clip_b32     0.001    0.864     0.868    0.295                 0.344
SAT+V_dinov2_base  0.001    0.863     0.858    0.299                 0.328
SAT+V_resnet50     0.001    0.853     0.851    0.277                 0